# 言語モデルの文法知識を最小対で調べる (BLiMP / JBLiMP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_llm/minimal_pairs.ipynb)

NLP クックブック 13-16 ／ 大規模言語モデル (LLM)

言語モデルが文法をどれくらい身に付けているかを調べたい。1 か所だけ違う「文法的な文」と「非文法的な文」の組 (最小対) を与え、文法的なほうに高い確率を付けるかを見る。

In [1]:
!pip install -q datasets sentencepiece

In [2]:
import json, urllib.request
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

url = "https://raw.githubusercontent.com/osekilab/JBLiMP/main/data/validated/validated_minimal_pairs.jsonl"
pairs = pd.DataFrame([json.loads(line) for line in urllib.request.urlopen(url).read().decode("utf-8").strip().split("\n")])
print(len(pairs), "組"); print(pairs["phenomenon"].value_counts().to_string())
print(pairs.iloc[0][["good_sentence", "bad_sentence", "paradigm"]].to_dict())

331 組
phenomenon
argument structure    140
verbal agreement       61
morphology             35
nominal structure      23
ellipsis               19
quantifiers            14
binding                13
island effects         11
filler-gap              9
NPI licensing           4
control/raising         2
{'good_sentence': '太郎が昨日花子に会った人を探しているのだ。', 'bad_sentence': '太郎が昨日会った人を探しているのは花子にだ。', 'paradigm': 'complex NP island'}


In [3]:
name = "rinna/japanese-gpt2-small"
tokenizer = AutoTokenizer.from_pretrained(name, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(name).eval()

@torch.no_grad()
def log_prob(sentence):
    """文全体の対数確率 (各トークンの対数確率の和) と、トークン数"""
    ids = tokenizer(sentence, return_tensors="pt")["input_ids"]
    ids = torch.cat([torch.tensor([[tokenizer.bos_token_id]]), ids], dim=1)
    logp = model(ids).logits[0, :-1].log_softmax(-1)
    return float(logp.gather(1, ids[0, 1:, None]).sum()), ids.shape[1] - 1

scores = [(log_prob(g), log_prob(b)) for g, b in zip(pairs["good_sentence"], pairs["bad_sentence"])]
pairs["good_lp"], pairs["good_len"] = [s[0][0] for s in scores], [s[0][1] for s in scores]
pairs["bad_lp"], pairs["bad_len"] = [s[1][0] for s in scores], [s[1][1] for s in scores]
pairs["correct"] = pairs["good_lp"] > pairs["bad_lp"]
print("全体の正解率:", round(pairs["correct"].mean(), 3), "(でたらめなら 0.5)")
print(pairs.groupby("phenomenon")["correct"].agg(["mean", "count"]).round(3).sort_values("mean").to_string())

全体の正解率: 0.779 (でたらめなら 0.5)
                     mean  count
phenomenon                      
control/raising     0.000      2
NPI licensing       0.500      4
verbal agreement    0.590     61
filler-gap          0.667      9
binding             0.692     13
argument structure  0.807    140
ellipsis            0.842     19
quantifiers         0.857     14
island effects      0.909     11
nominal structure   0.913     23
morphology          0.943     35


In [4]:
# 長さの偏り: 文の確率はトークンが増えるほど下がるので、短いほうの文が有利になる
pairs["diff"] = pairs["good_len"] - pairs["bad_len"]
pairs["length"] = np.select([pairs["diff"] < 0, pairs["diff"] == 0], ["文法的な文のほうが短い", "同じ長さ"], "文法的な文のほうが長い")
print(pairs.groupby("length")["correct"].agg(["mean", "count"]).round(3).to_string())

# 対処: トークン数で割った平均対数確率で比べる
pairs["correct_norm"] = pairs["good_lp"] / pairs["good_len"] > pairs["bad_lp"] / pairs["bad_len"]
print("長さで正規化した正解率:", round(pairs["correct_norm"].mean(), 3))
print(pairs.groupby("length")["correct_norm"].mean().round(3).to_string())

              mean  count
length                   
同じ長さ         0.735    147
文法的な文のほうが短い  0.916    119
文法的な文のほうが長い  0.631     65
長さで正規化した正解率: 0.743
length
同じ長さ           0.735
文法的な文のほうが短い    0.622
文法的な文のほうが長い    0.985


In [5]:
# 間違えた組を見る
for r in pairs[~pairs["correct"]].head(5).itertuples():
    print(f"[{r.phenomenon}] ○ {r.good_sentence} ({r.good_lp:.1f}, {r.good_len} トークン)\n            × {r.bad_sentence} ({r.bad_lp:.1f}, {r.bad_len} トークン)")

[island effects] ○ 初めて会う人がけなすのは誰をですか？ (-64.2, 13 トークン)
            × 初めて会う人がけなすのは誰にですか？ (-61.7, 13 トークン)
[ellipsis] ○ 読んだ学生がうんざりしたのはどの論文にですか？ (-71.7, 14 トークン)
            × 読んだ学生がどの論文にうんざりしたの？ (-63.3, 14 トークン)
[binding] ○ 初めてそいつに会った人が太郎をけなしたの？ (-68.6, 15 トークン)
            × 初めてそいつに会った人が誰をけなしたの？ (-67.3, 15 トークン)
[binding] ○ 初めて会った人が太郎をけなしたの？ (-55.2, 12 トークン)
            × 初めて会った人が誰をけなしたの？ (-52.9, 12 トークン)
[argument structure] ○ このドアが太郎によって開けられた。 (-53.5, 9 トークン)
            × このドアが太郎に開けられた。 (-52.8, 9 トークン)


- 最小対による評価は、言語モデルに追加の学習をさせずに文法知識を測れる。英語の BLiMP、日本語の JBLiMP などがあり、現象ごと (一致、束縛、島の制約など) に得意・不得意が分かる。
- 文全体の確率で比べると、トークン数の少ない文が有利になる。最小対の 2 文でトークン数が違う場合、モデルが文法ではなく長さで選んでいるだけかもしれない。データセットによっては、この偏りだけで高い正解率が出る。
- 長さで正規化すれば解決する、とは限らない。この実験では、正規化すると今度は長い文が有利になり、偏りの向きが逆になっただけだった。トークン数が同じ組だけで評価する、長さの条件ごとに正解率を分けて報告する、といった確認が確実。
- 正解率が高くても、その現象を「理解している」とは限らない。語の頻度や共起のような表面的な手がかりで解ける組もある。
- 指示に従う LLM なら「どちらが自然か」を直接尋ねることもできるが、確率を比べる方法とは結果が一致しないことがある。